In [1]:
import requests
import pandas as pd
import re
import os
import time
from collections import Counter

# ============================================================
# CONFIG
# ============================================================

API = "https://infini-news.uni-graz.at/api/v1"
INDEX = "ccnews"

HUMAN_PATH = "/kaggle/input/datasets/tanjimshaharia/human-articles-100/human_articles_100.csv"

OUT_PATH = "/kaggle/working/selected_english_100.csv"
CHECKPOINT_PATH = "/kaggle/working/selected_english_100_checkpoint.csv"

# Search only recent-ish years to reduce API work.
# Change to None if you want the entire 2016-2026 corpus.
YEAR_MIN = 2020
YEAR_MAX = 2026

# Number of candidate documents retrieved per query
MAX_CANDIDATES_PER_QUERY = 8

# ============================================================
# LOAD YOUR EXACT 100 TOPICS
# ============================================================

human = pd.read_csv(HUMAN_PATH)

topics = human[["topic_id", "topic"]].copy()
topics["topic_id"] = topics["topic_id"].astype(int)
topics = topics.sort_values("topic_id").reset_index(drop=True)

print("Topics loaded:", len(topics))

# ============================================================
# HELPERS
# ============================================================

session = requests.Session()
session.headers.update({
    "Content-Type": "application/json",
    "User-Agent": "INFINI-News-Bangla-Research/1.0"
})

STOPWORDS = {
    "in", "the", "and", "of", "for", "to", "on", "with",
    "in", "a", "an", "bangladesh", "bangladeshi"
}

def clean_text(x):
    if x is None:
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def meaningful_words(topic):
    words = re.findall(r"[A-Za-z]+", topic.lower())
    return [w for w in words if w not in STOPWORDS and len(w) >= 3]

def make_queries(topic):
    """
    Make a few exact-string queries, from most specific
    to broader. INFINI searches exact strings.
    """
    topic_clean = clean_text(topic)

    queries = []

    # 1. Exact topic
    queries.append(topic_clean)

    words = meaningful_words(topic_clean)

    # 2. Remove Bangladesh / stopwords
    if len(words) >= 2:
        queries.append(" ".join(words))

    # 3. First two important terms
    if len(words) >= 3:
        queries.append(" ".join(words[:2]))

    # 4. Last two important terms
    if len(words) >= 3:
        queries.append(" ".join(words[-2:]))

    # Remove duplicates while preserving order
    out = []
    seen = set()
    for q in queries:
        q = q.strip()
        if q and q.lower() not in seen:
            out.append(q)
            seen.add(q.lower())

    return out


def find_segments(query):
    payload = {
        "query": query,
        "index": INDEX,
        "year_min": YEAR_MIN,
        "year_max": YEAR_MAX
    }

    r = session.post(
        f"{API}/find",
        json=payload,
        timeout=60
    )
    r.raise_for_status()
    return r.json()


def get_document(query, shard, rank):
    payload = {
        "query": query,
        "index": INDEX,
        "year_min": YEAR_MIN,
        "year_max": YEAR_MAX,
        "s": int(shard),
        "rank": int(rank),
        "max_ctx_len": 2000
    }

    r = session.post(
        f"{API}/get_doc",
        json=payload,
        timeout=60
    )

    if r.status_code != 200:
        return None

    try:
        return r.json()
    except:
        return None


def extract_passage(text, needle_offset, min_words=100, max_words=200):
    """
    Extract approximately 150 words centered around the
    matched query, while keeping it contiguous.
    """

    text = clean_text(text)

    if not text:
        return ""

    words = text.split()

    if len(words) <= max_words:
        return text

    # Convert character offset into approximate word index
    before = text[:max(0, int(needle_offset))]
    center = len(before.split())

    # Target ~150 words
    target = 150

    start = max(0, center - target // 2)
    end = min(len(words), start + target)

    # Shift window if near the end
    if end - start < target:
        start = max(0, end - target)

    passage = " ".join(words[start:end])

    # Make sure we have at least 100 words
    if len(passage.split()) < min_words:
        start = max(0, center - 100)
        end = min(len(words), center + 100)
        passage = " ".join(words[start:end])

    return passage


def english_score(doc, query, topic):
    """
    Rank candidate articles.
    Main requirement: English.
    Then favor topic terms appearing in title/snippet.
    """

    metadata = doc.get("metadata", {}) or {}

    language = str(metadata.get("language") or "").lower()

    if language not in {"eng", "en", "english"}:
        return -999

    title = clean_text(metadata.get("title", "")).lower()
    text = clean_text(doc.get("text", "")).lower()

    topic_words = meaningful_words(topic)

    score = 100

    # Strong preference for title relevance
    for w in topic_words:
        if w in title:
            score += 12
        if w in text:
            score += 3

    # Penalize extremely tiny snippets
    wc = len(text.split())

    if wc < 70:
        score -= 30
    elif wc < 100:
        score -= 10

    return score


def collect_candidates(topic):
    candidates = []
    seen_docs = set()

    queries = make_queries(topic)

    for query in queries:

        try:
            result = find_segments(query)
        except Exception as e:
            print("   find error:", str(e)[:120])
            continue

        segments = result.get("segment_by_shard", [])
        shard_years = result.get("shard_years", [])

        # Collect a limited number of ranks from each matching shard.
        # This avoids downloading hundreds/thousands of documents.
        retrieved = 0

        for s, segment in enumerate(segments):

            if retrieved >= MAX_CANDIDATES_PER_QUERY:
                break

            if not isinstance(segment, list) or len(segment) != 2:
                continue

            start, end = segment

            if end <= start:
                continue

            # Sample a few ranks from this shard.
            # First, middle, and another position where possible.
            possible = [start]

            if end - start > 2:
                possible.append(start + (end - start) // 2)

            if end - start > 1:
                possible.append(end - 1)

            for rank in possible:

                if retrieved >= MAX_CANDIDATES_PER_QUERY:
                    break

                try:
                    doc = get_document(query, s, rank)
                except Exception:
                    continue

                if not doc:
                    continue

                metadata = doc.get("metadata", {}) or {}

                # English only
                language = str(
                    metadata.get("language") or ""
                ).lower()

                if language not in {"eng", "en", "english"}:
                    continue

                url = metadata.get("url")
                title = metadata.get("title")

                # Avoid same document appearing through multiple queries
                doc_key = (
                    str(url).lower()
                    if url
                    else (
                        str(metadata.get("hostname")).lower(),
                        str(title).lower(),
                        str(metadata.get("date")).lower()
                    )
                )

                if doc_key in seen_docs:
                    continue

                seen_docs.add(doc_key)

                score = english_score(doc, query, topic)

                candidates.append({
                    "query_used": query,
                    "score": score,
                    "doc": doc
                })

                retrieved += 1

        # If we already have enough decent candidates,
        # don't waste more API calls.
        if len(candidates) >= 5:
            candidates = sorted(
                candidates,
                key=lambda x: x["score"],
                reverse=True
            )

            if candidates[0]["score"] >= 110:
                break

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return candidates


# ============================================================
# RESUME FROM CHECKPOINT IF AVAILABLE
# ============================================================

if os.path.exists(CHECKPOINT_PATH):
    existing = pd.read_csv(CHECKPOINT_PATH)
    done_ids = set(existing["topic_id"].astype(int))
    results = existing.to_dict("records")

    print(
        f"Resuming: {len(done_ids)} topics already completed."
    )
else:
    done_ids = set()
    results = []

print("=" * 70)
print("Starting INFINI-NEWS retrieval")
print("=" * 70)


# ============================================================
# MAIN LOOP
# ============================================================

for _, row in topics.iterrows():

    topic_id = int(row["topic_id"])
    topic = str(row["topic"])

    if topic_id in done_ids:
        continue

    print(f"\n[{topic_id}/100] {topic}")

    try:
        candidates = collect_candidates(topic)

        if not candidates:
            print("   ❌ No English candidate found.")
            continue

        best = candidates[0]
        doc = best["doc"]

        metadata = doc.get("metadata", {}) or {}

        text = clean_text(doc.get("text", ""))

        passage = extract_passage(
            text,
            doc.get("needle_offset", 0),
            min_words=100,
            max_words=200
        )

        wc = len(passage.split())

        # Basic validity check
        if wc < 80:
            print(
                f"   ⚠️ Passage too short ({wc} words), skipping."
            )
            continue

        result = {
            "topic_id": topic_id,
            "topic": topic,
            "article_id": doc.get("doc_ix"),
            "headline": metadata.get("title"),
            "english_passage": passage,
            "english_word_count": wc,
            "publication_date": metadata.get("date"),
            "source": metadata.get("sitename"),
            "hostname": metadata.get("hostname"),
            "url": metadata.get("url"),
            "search_query": best["query_used"],
            "search_score": best["score"],
            "source_corpus": "INFINI-NEWS / CC-News"
        }

        results.append(result)
        done_ids.add(topic_id)

        print(
            f"   ✓ {wc} words | "
            f"{metadata.get('sitename')} | "
            f"{metadata.get('date')} | "
            f"query='{best['query_used']}'"
        )

        # Checkpoint after EVERY successful topic
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df = checkpoint_df.sort_values("topic_id")
        checkpoint_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        time.sleep(0.15)

    except Exception as e:
        print(
            f"   ❌ ERROR: {type(e).__name__}: {str(e)[:200]}"
        )

# ============================================================
# FINAL SAVE
# ============================================================

final_df = pd.DataFrame(results)

if len(final_df) > 0:
    final_df = final_df.sort_values("topic_id")

    # Keep exactly one article per topic
    final_df = final_df.drop_duplicates(
        subset=["topic_id"],
        keep="first"
    )

    final_df.to_csv(
        OUT_PATH,
        index=False
    )

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print("Rows:", len(final_df))
print("Missing:", 100 - len(final_df))

if len(final_df) > 0:
    missing = sorted(
        set(topics["topic_id"]) -
        set(final_df["topic_id"])
    )

    print("Missing topic IDs:", missing)

    print("\nWord count:")
    print(final_df["english_word_count"].describe())

    print("\nSaved:")
    print(OUT_PATH)

    print("\nCheckpoint:")
    print(CHECKPOINT_PATH)

Topics loaded: 100
Starting INFINI-NEWS retrieval

[1/100] Traffic congestion in Dhaka
   ✓ 150 words | The Daily Observer | 2022-04-02 | query='Traffic congestion in Dhaka'

[2/100] Air pollution in Bangladesh
   ✓ 150 words | Mongabay Environmental News | 2022-06-16 | query='Air pollution in Bangladesh'

[3/100] Water pollution in Bangladesh
   ✓ 198 words | SFGate | 2020-01-08 | query='water pollution'

[4/100] Plastic waste management
   ✓ 150 words | fnbherald.com | 2020-01-22 | query='Plastic waste management'

[5/100] Flooding in Bangladesh
   ✓ 111 words | ReliefWeb | 2020-07-17 | query='Flooding in Bangladesh'

[6/100] River erosion in Bangladesh
   ✓ 150 words | CBC | 2020-02-22 | query='river erosion'

[7/100] Coastal erosion
   ✓ 150 words | Hulldailymail | 2020-01-27 | query='Coastal erosion'

[8/100] Rising sea levels
   ✓ 150 words | The Pacific Islands are drowning under rising sea levels. These stunning photos show their precarious way of life., Business Insider | 2020

In [1]:
# ============================================
# STAGE 3 — BUET BanglaT5: English → Bangla
# FAST BATCHED VERSION
# ============================================

!pip -q install transformers sentencepiece accelerate

import os
import re
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --------------------------------------------
# PATHS
# --------------------------------------------
INPUT_PATH = "/kaggle/input/datasets/tanjimshaharia/selected-100-csv/selected_english_100.csv"

OUTPUT_PATH = "/kaggle/working/mt_bangla_articles.csv"
CHECKPOINT_PATH = "/kaggle/working/mt_bangla_checkpoint.csv"

MODEL_NAME = "csebuetnlp/banglat5_nmt_en_bn"

# --------------------------------------------
# DEVICE
# --------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------
# LOAD DATA
# --------------------------------------------
df = pd.read_csv(INPUT_PATH)

df = df.sort_values("topic_id").reset_index(drop=True)

print("English texts:", len(df))

# --------------------------------------------
# LOAD MODEL
# --------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

print("BanglaT5 loaded!")

# --------------------------------------------
# RESUME CHECKPOINT
# --------------------------------------------
if os.path.exists(CHECKPOINT_PATH):

    results = pd.read_csv(CHECKPOINT_PATH)

    completed_ids = set(
        results["topic_id"].astype(int)
    )

    print("Already completed:", len(completed_ids))

else:

    results = pd.DataFrame()

    completed_ids = set()

# --------------------------------------------
# TRANSLATION FUNCTION
# --------------------------------------------
def translate_batch(texts):

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=2,
            early_stopping=True
        )

    translations = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    return [x.strip() for x in translations]


# --------------------------------------------
# PREPARE REMAINING DATA
# --------------------------------------------
remaining = df[
    ~df["topic_id"].isin(completed_ids)
].copy()

print("Remaining:", len(remaining))

# --------------------------------------------
# BATCH TRANSLATION
# --------------------------------------------
BATCH_SIZE = 8

for start in range(0, len(remaining), BATCH_SIZE):

    batch = remaining.iloc[
        start:start + BATCH_SIZE
    ]

    texts = batch["english_passage"].astype(str).tolist()

    try:

        translations = translate_batch(texts)

        batch_results = []

        for (_, row), bangla in zip(
            batch.iterrows(),
            translations
        ):

            english_words = len(
                str(row["english_passage"]).split()
            )

            bangla_words = len(
                bangla.split()
            )

            # Basic quality check
            if bangla_words < 40:
                status = "suspicious"
            else:
                status = "success"

            batch_results.append({
                "topic_id": int(row["topic_id"]),
                "topic": row["topic"],
                "article_id": row.get("article_id", ""),
                "headline": row.get("headline", ""),
                "english_passage": row["english_passage"],
                "english_word_count": english_words,
                "mt_bangla": bangla,
                "mt_word_count": bangla_words,
                "mt_model": MODEL_NAME,
                "mt_status": status
            })

        batch_df = pd.DataFrame(batch_results)

        results = pd.concat(
            [results, batch_df],
            ignore_index=True
        )

        results = (
            results
            .drop_duplicates("topic_id", keep="last")
            .sort_values("topic_id")
            .reset_index(drop=True)
        )

        # CHECKPOINT
        results.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        print(
            f"Translated "
            f"{min(start + BATCH_SIZE, len(remaining))}"
            f"/{len(remaining)} remaining"
        )

    except Exception as e:

        print(
            f"❌ Batch starting at {start} failed:",
            e
        )

# --------------------------------------------
# FINAL SAVE
# --------------------------------------------
results = (
    results
    .sort_values("topic_id")
    .reset_index(drop=True)
)

results.to_csv(
    OUTPUT_PATH,
    index=False
)

# --------------------------------------------
# SUMMARY
# --------------------------------------------
print("\n====================================")
print("DONE")
print("Translations:", len(results))
print("Successful:", (results["mt_status"] == "success").sum())
print("Suspicious:", (results["mt_status"] == "suspicious").sum())
print("Output:", OUTPUT_PATH)
print("====================================")

print("\nWord count:")
print(
    results[
        ["english_word_count", "mt_word_count"]
    ].describe()
)

Device: cuda
GPU: Tesla T4
English texts: 100


config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 1.11MB            

spiece.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

BanglaT5 loaded!
Remaining: 100
Translated 8/100 remaining
Translated 16/100 remaining
Translated 24/100 remaining
Translated 32/100 remaining
Translated 40/100 remaining
Translated 48/100 remaining
Translated 56/100 remaining
Translated 64/100 remaining
Translated 72/100 remaining
Translated 80/100 remaining
Translated 88/100 remaining
Translated 96/100 remaining
Translated 100/100 remaining

DONE
Translations: 100
Successful: 97
Suspicious: 3
Output: /kaggle/working/mt_bangla_articles.csv

Word count:
       english_word_count  mt_word_count
count          100.000000     100.000000
mean           150.340000      87.600000
std             10.056628      15.931672
min            111.000000       6.000000
25%            150.000000      83.750000
50%            150.000000      88.000000
75%            150.000000      96.000000
max            200.000000     120.000000


In [6]:
# ============================================================
# NLLB-200 600M — ENGLISH → BANGLA
# GPU / Tesla T4
# ============================================================

!pip -q install -U transformers sentencepiece accelerate

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# 1. FILES
# ------------------------------------------------------------

INPUT_FILE = "/kaggle/input/datasets/tanjimshaharia/selected-100-csv/selected_english_100.csv"
OUTPUT_FILE = "/kaggle/working/mtt_bangla_articles.csv"

# ------------------------------------------------------------
# 2. MODEL
# ------------------------------------------------------------

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 3. LOAD TOKENIZER + MODEL
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="eng_Latn"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

# Force Bengali output
forced_bos_token_id = tokenizer.convert_tokens_to_ids(
    "ben_Beng"
)

print("Model loaded.")
print("Forced Bengali token:", forced_bos_token_id)

# ------------------------------------------------------------
# 4. LOAD ENGLISH DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("\nEnglish dataset:", df.shape)

df = df[
    [
        "topic_id",
        "topic",
        "article_id",
        "headline",
        "english_passage",
        "english_word_count",
        "publication_date",
        "source",
        "hostname",
        "url",
        "search_query",
        "search_score",
        "source_corpus"
    ]
].copy()

df["english_passage"] = (
    df["english_passage"]
    .fillna("")
    .astype(str)
)

# ------------------------------------------------------------
# 5. TRANSLATION
# ------------------------------------------------------------

BATCH_SIZE = 8

translations = []

for start in range(0, len(df), BATCH_SIZE):

    batch = df.iloc[start:start + BATCH_SIZE]

    texts = batch["english_passage"].tolist()

    print(
        f"Translating {start + 1}-{start + len(batch)} "
        f"of {len(df)}"
    )

    # NLLB does not need a natural-language prompt.
    # The source/target language tokens control translation.
    encoded = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        output_ids = model.generate(
            **encoded,

            # Explicitly force Bengali
            forced_bos_token_id=forced_bos_token_id,

            # More reliable translation than greedy decoding
            num_beams=5,

            # Allow enough room for the Bengali output
            max_length=300,

            # Prevent pathological repetition
            repetition_penalty=1.05,

            # Slightly discourage unfinished generations
            length_penalty=1.0,

            early_stopping=True
        )

    batch_translations = tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True
    )

    translations.extend(batch_translations)

# ------------------------------------------------------------
# 6. SAVE
# ------------------------------------------------------------

df["mt_bangla"] = translations

df["mt_word_count"] = (
    df["mt_bangla"]
    .str.split()
    .str.len()
)

df["mt_model"] = MODEL_NAME

df["mt_translation_method"] = (
    "NLLB-200 eng_Latn→ben_Beng, "
    "beam_search_5"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. DONE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("NLLB TRANSLATION COMPLETE")
print("=" * 60)

print("Rows:", len(df))
print("Output:", OUTPUT_FILE)

print("\nWord count summary:")
print(df["mt_word_count"].describe())

print("\nFirst 3 translations:\n")

for _, row in df.head(3).iterrows():

    print("-" * 60)
    print("TOPIC:", row["topic"])
    print("ENGLISH:")
    print(row["english_passage"][:500])

    print("\nBANGLA:")
    print(row["mt_bangla"])

Device: cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Model loaded.
Forced Bengali token: 256026

English dataset: (100, 13)
Translating 1-8 of 100
Translating 9-16 of 100
Translating 17-24 of 100
Translating 25-32 of 100
Translating 33-40 of 100
Translating 41-48 of 100
Translating 49-56 of 100
Translating 57-64 of 100
Translating 65-72 of 100
Translating 73-80 of 100
Translating 81-88 of 100
Translating 89-96 of 100
Translating 97-100 of 100

NLLB TRANSLATION COMPLETE
Rows: 100
Output: /kaggle/working/mtt_bangla_articles.csv

Word count summary:
count    100.000000
mean     115.700000
std       21.832929
min       49.000000
25%      102.500000
50%      121.000000
75%      129.250000
max      173.000000
Name: mt_word_count, dtype: float64

First 3 translations:

------------------------------------------------------------
TOPIC: Traffic congestion in Dhaka
ENGLISH:
Dhaka’s traffic in a big mess | | Traffic congestion in Dhaka has increased tremendously. Various surveys have shown that the average loss per day due to traffic jams is Rs 30